In [ ]:
from pprint import pprint

from mongoengine import connect
from fastapi_filter.contrib.mongoengine import Filter

from app.core.config import settings
from app.schemas.component_schema import CpuSchema
from app.filters.component_filters import CpuFilters, CpuDocument

connect(db=settings.DATABASE, host=settings.MONGO_CONNECTION_STRING)

MongoClient(host=['ac-jmroqqw-shard-00-01.hn8twum.mongodb.net:27017', 'ac-jmroqqw-shard-00-02.hn8twum.mongodb.net:27017', 'ac-jmroqqw-shard-00-00.hn8twum.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, retrywrites=True, w='majority', appname='PConfCluster', authsource='admin', replicaset='atlas-9wgods-shard-0', tls=True, read_preference=Primary(), uuidrepresentation=3, driver=DriverInfo(name='MongoEngine', version='0.29.1', platform=None))

In [ ]:

components = CpuFilter(manufacturer="amd", cores__gt=9, cores__lt=13).filter(CpuDocument.objects())

In [ ]:
for c in components:
    print(c.to_json())

In [ ]:
dir(CpuSchema)

In [ ]:
# pprint(CpuSchema.model_json_schema())


schema = CpuSchema.model_json_schema()

pprint(
  schema["properties"]["cores"]
)

### Не поганий базовий варіант, проте підтримує тільки анотації типу `str`

In [ ]:

from types import new_class
from typing import TypeAlias, ClassVar

from fastapi_filter.contrib.mongoengine import Filter

StrOrNone: TypeAlias = str | None


def filter_generator(fields: dict[str, list[str]]):

    annotations = {}
    default_values = {}

    for f_name, f_values in fields.items():
        for f_value in f_values:

            if f_value != "exact":
                f_name = f"{f_name}_{f_value}"

            annotations[f_name] = StrOrNone
            default_values[f_name] = None

    # Функція для ініціалізації простору імен класу
    def class_namespace(ns: dict):
        ns["__module__"] = __name__

        annotations["Constants"] = ClassVar[type]
        ns["__annotations__"] = annotations

        ns.update(default_values)

        class Constants(Filter.Constants):
            model = CpuDocument
        ns["Constants"] = Constants

    new_filter_class = new_class(
        name="CustomFilterClass",
        bases=(Filter,),
        kwds={},
        exec_body=class_namespace,
    )

    return new_filter_class


fields = {
    "id": ["exact", "in"],
    "model": ['exact', "in", "contains"],
    "manufacturer": ['exact', 'contains'],
}

CpuFilter = filter_generator(fields)


components = CpuFilter(manufacturer="amd").filter(CpuDocument.objects())

print(CpuFilter.__annotations__)
print(components)
print(CpuFilter.model_json_schema())




# class CpuFilter:
#     model = CpuDocument

#     fields = {
#         "id": ["exact", "in"],
#         "model": ['exact'],
#         "manufacturer": ['exact', 'contains'],
#     }


In [ ]:
# Додаємо підклас Constants
class Constants(Filter.Constants):
    model = CpuDocument


annotations = {"a": int}
default_values = {"a": 1}

Fields = type(
    "Fields",
    (),
    {"__annotations__": annotations, **default_values, "Constants": Constants}
)

class A(Fields, Filter):
    pass


### Спробую додати підтримку інших типів даних. Для цього буду передавати сам тип у списку `fields`

In [2]:
from types import new_class
from typing import Optional, TypeAlias, ClassVar

from fastapi_filter.contrib.mongoengine import Filter
from pydantic_extra_types.mongo_object_id import MongoObjectId
from mongoengine import fields as f, Document


def get_annotations_from_model(model: Document):
    annotations = CpuDocument.__annotations__
    annotations["id"] = MongoObjectId
    for cls in CpuDocument.__mro__:
        if cls is Document:
            break
        annotations.update(cls.__annotations__)
    return annotations


def filter_generator(model: Document, fields: dict[str, list[str]]):

    model_fields = get_annotations_from_model(model)
    print(model_fields)
    
    annotations = {}
    default_values = {}

    for f_name, f_values in fields.items():
        for f_value in f_values:

            f_name_full = f_name

            if f_value != "exact":
                f_name_full = f"{f_name}__{f_value}"

            if f_value == "in":
                annotations[f_name_full] = Optional[list[model_fields[f_name]]]
                default_values[f_name_full] = None
                continue

            annotations[f_name_full] = Optional[model_fields[f_name]]
            default_values[f_name_full] = None

    # print(annotations)
    # print(default_values)
    # for annotation, default_value in zip(annotations, default_values):
    #     print(f"{annotation:25}{default_value}")

    # Функція для ініціалізації простору імен класу
    def class_namespace(ns: dict):
        
        ns["__module__"] = __name__

        annotations["Constants"] = ClassVar[type]
        ns["__annotations__"] = annotations

        ns.update(default_values)

        class Constants(Filter.Constants): ...
        Constants.model = model

        ns["Constants"] = Constants

    new_filter_class = new_class(
        name="CustomFilterClass",
        bases=(Filter,),
        kwds={},
        exec_body=class_namespace,
    )

    return new_filter_class


fields = {
    "id": ["exact", "in"],
    "model": ['exact', "in", "contains"],
    "manufacturer": ['exact', "in", 'contains'],
}

CpuFilter = filter_generator(CpuDocument, fields)


components = CpuFilter(manufacturer__in="amd,intel", id="68266380ffe1e50a2f71142a").filter(CpuDocument.objects())

# print(CpuFilter.__annotations__)
print(components)
# print(CpuFilter.model_json_schema())


{'socket': <class 'str'>, 'ram_types': <class 'str'>, 'igp': <class 'bool'>, 'tdp': <class 'int'>, 'tdp_max': <class 'int'>, 'cores': <class 'int'>, 'threads': <class 'int'>, 'id': <class 'pydantic_extra_types.mongo_object_id.MongoObjectId'>, 'manufacturer': <class 'str'>, 'model': <class 'str'>}
[<CpuDocument: CpuDocument object>]


In [ ]:

field_name = "model"

for name, field in CpuDocument._fields.items():
    print(name, field, "→", type(field.to_python(field.default)))


In [ ]:
CpuDocument.__mro__

In [ ]:
from mongoengine import Document

annotations = CpuDocument.__annotations__

for cls in CpuDocument.__mro__:
    if cls is Document:
        break
    annotations.update(cls.__annotations__)

annotations


# print(cls.__class__.__name__, Document.__class__.__name__)

In [ ]:
CpuDocument._fields

In [ ]:
from app.filters.filter_utils import create_filter_class

DynamicCpuFilter = create_filter_class(
    model=CpuDocument,
    fields={
        "id": ["exact", "in"],
        "model": ["exact", "in", "contains"],
        "manufacturer": ["exact", "in", "contains"],
    },
)

In [9]:
res = DynamicCpuFilter(manufacturer__in="amd", id="68266380ffe1e50a2f71142a").filter(CpuDocument.objects())

In [10]:
len(res)

1

In [7]:
# CpuFilters(manufacturer="AMD").filter(CpuDocument.objects().limit(2))
CpuFilters(manufacturer="AMD").filter(CpuDocument.objects()[:2])

[<CpuDocument: CpuDocument object>, <CpuDocument: CpuDocument object>]